---
title: "Wikidata Item Profile"
format:
  html:
    toc: false
    code-fold: true
    code-summary: "Show code"
execute:
  echo: false
  warning: false
  error: false
jupyter: python3
---

# Wikidata Item Profile

This notebook-backed page queries Wikidata with SPARQL and renders a styled HTML profile.

To profile another item, change the `item_id` value in Cell 3 and re-render the site.

https://www.sprengel-museum.de/ausstellungen/aktuell/abenteuer-abstraktion 		



In [1]:
from IPython.display import HTML, Markdown
from wikidata_profile import (
    build_statement_query,
    fetch_sparql_bindings,
    properties_from_bindings,
    render_profile_html,
)

In [2]:
item_id = "Q138572982"
item_id = item_id.strip().upper()

if not item_id.startswith("Q") or not item_id[1:].isdigit():
    raise ValueError("item_id must be a Wikidata Q-id like Q42 or Q138547468")

query = build_statement_query(item_id)
Markdown(
    f"## SPARQL Query Used for {item_id}\n```sparql\n" + query + "\n```"
 )

## SPARQL Query Used for Q138572982
```sparql
SELECT ?property ?propertyLabel ?value ?valueLabel WHERE {
  BIND(wd:Q138572982 AS ?item)
  ?item ?p ?statement .
  ?property wikibase:claim ?p .
  ?statement ?ps ?value .
  ?property wikibase:statementProperty ?ps .

  SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }
}
ORDER BY ?propertyLabel
```

In [3]:
bindings = fetch_sparql_bindings(query)
properties = properties_from_bindings(bindings)
len(bindings)

17

In [4]:
HTML(render_profile_html(item_id, properties))

## Visualizing the Wikidata Item as a Graph
The following cell renders a graph visualization of the relationships for the selected Wikidata item. This helps to see how the item is connected to other entities via its properties.

In [ ]:
import networkx as nx
import plotly.graph_objects as go
import math
from IPython.display import display, HTML

G = nx.DiGraph()
central_label = bindings[0].get('itemLabel', {}).get('value', item_id) if bindings else item_id

for b in bindings:
    subj = b.get('itemLabel', {}).get('value', item_id)
    prop = b.get('propertyLabel', {}).get('value', '')
    obj  = b.get('valueLabel',  {}).get('value', b.get('value', {}).get('value', ''))
    if obj and prop:
        G.add_edge(subj, obj, label=prop)

if G.number_of_edges() == 0:
    display(HTML("<p style='color:red;'>Keine Graphdaten verfügbar.</p>"))
else:
    # Stern-Layout: Zentralknoten in der Mitte, alle anderen im Kreis
    non_center = [n for n in G.nodes() if n != central_label]
    pos = {central_label: (0, 0)}
    for i, node in enumerate(non_center):
        angle = 2 * math.pi * i / len(non_center)
        pos[node] = (math.cos(angle) * 2, math.sin(angle) * 2)

    edge_traces = []
    annotations = []

    for u, v, data in G.edges(data=True):
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        mx, my = (x0 + x1) / 2, (y0 + y1) / 2

        edge_traces.append(go.Scatter(
            x=[x0, x1, None], y=[y0, y1, None],
            mode='lines',
            line=dict(width=1.5, color='#ffaaaa'),
            hoverinfo='none',
            showlegend=False
        ))

        annotations.append(dict(
            ax=x0 + 0.5*(x1-x0), ay=y0 + 0.5*(y1-y0),
            x=x0  + 0.6*(x1-x0), y=y0  + 0.6*(y1-y0),
            xref='x', yref='y', axref='x', ayref='y',
            showarrow=True, arrowhead=3, arrowsize=1.3,
            arrowwidth=1.5, arrowcolor='#cc0000'
        ))

        annotations.append(dict(
            x=mx, y=my, xref='x', yref='y',
            text=f"<i>{data.get('label','')}</i>",
            showarrow=False,
            font=dict(size=10, color='#990000'),
            bgcolor='rgba(255,255,255,0.85)',
            borderpad=2,
        ))

    node_x, node_y, node_text, node_color, node_size = [], [], [], [], []

    for node in G.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        node_text.append(node)
        is_central = (node == central_label)
        node_color.append('#cc0000' if is_central else '#ff6666')
        node_size.append(32 if is_central else 20)

    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers+text',
        text=node_text,
        textposition='top center',
        textfont=dict(size=11, color='#1a1a1a'),
        hoverinfo='text',
        marker=dict(
            size=node_size,
            color=node_color,
            line=dict(width=2, color='white'),
        ),
        showlegend=False
    )

    fig = go.Figure(data=[*edge_traces, node_trace])
    fig.update_layout(
        title=dict(
            text=f"Wikidata-Graph · {central_label} ({item_id})",
            font=dict(size=15, color='#cc0000'), x=0.5
        ),
        showlegend=False,
        hovermode='closest',
        margin=dict(b=30, l=30, r=30, t=50),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-3.2, 3.2]),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-3.2, 3.2]),
        plot_bgcolor='white',
        paper_bgcolor='white',
        height=750,
        annotations=annotations,
    )

    fig
